<h1 style=\"text-align: center; font-size: 50px;\"> <h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1> </h1>

📘 Project Overview: 
 This notebook demonstrates a modular architecture for answering natural language questions 
 over one or more feedback documents using only local and open-source models (e.g., LLaMA.cpp).
 The system processes long documents chunk-by-chunk and synthesizes a final answer using a multi-step LLM workflow.

# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 45.6 ms, sys: 2.28 ms, total: 47.9 ms
Wall time: 1.48 s


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

# Start Execution

In [3]:
# ─────── Standard Library Imports ───────
import os  # OS-level utilities like path and environment operations
import sys  # Access to interpreter variables and runtime configuration
import time  # Time-related functions

# Extend sys.path to include parent directory for local module resolution
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# ─────── Local Application Imports ───────
from src.utils import (  # Core utilities for logging, LLM interaction, and schema generation
    display_image,
    get_model_path,
    get_response_from_llm,
    json_schema_from_type,
    load_config,
    log_timing,
    logger,
)


In [4]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [5]:
TOPIC: str = "Focus Flow"  
QUESTION: str = "Which poeple provided the feedback?"

# Install and Import Libraries

In [7]:
from __future__ import annotations  # Enables postponed evaluation of type annotations (PEP 563)

# ─────── Standard Library Imports ───────
import base64  # Encoding and decoding binary data
import functools  # Functional programming utilities like lru_cache, partial, etc.
import json  # JSON serialization and deserialization
import logging  # Logging framework
import multiprocessing  # Parallel execution using subprocesses
import os  # OS-level utilities
import shutil  # File and directory operations
import sys  # Access to runtime environment and system-specific parameters
import time  # Time tracking and delays
import warnings  # Warning control and filtering
from collections import namedtuple  # Lightweight object types
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Static typing annotations

# ─────── Third-Party Package Imports ───────
import mlflow  # Model tracking and serving framework
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
from mlflow.models.signature import ModelSignature  # MLflow model signature
from mlflow.types.schema import Schema, ColSpec  # MLflow schema types
import yaml  # YAML file parsing
from IPython.display import HTML, display, Markdown  # Rich output formatting in Jupyter environments
from tqdm import tqdm  # Progress bar for loops

# ─────── LangChain Core & Community Imports ───────
from langchain.docstore.document import Document  # Document abstraction
from langchain.text_splitter import RecursiveCharacterTextSplitter  # Intelligent text splitting
from langchain_community.document_loaders import (  # Document loaders for different file types
    CSVLoader,
    PyPDFLoader,
    TextLoader,
    UnstructuredExcelLoader,
    UnstructuredMarkdownLoader,
    UnstructuredWordDocumentLoader,
)
from langchain_community.llms import LlamaCpp  # Integration for running LlamaCpp locally

# ─────── LangGraph Imports ───────
from langgraph.graph import END, START, StateGraph  # Constructs and controls stateful agent graphs

# ─────── Local Application-Specific Imports ───────
from src.simple_kv_memory import SimpleKVMemory  # In-memory store for agent state

# ─────── New Universal MLflow Structure ───────
from src.mlflow import Logger  # Universal Logger for models-from-code registration

from IPython import get_ipython

# Configure Settings

In [8]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [9]:
# ─────── Path Configuration ───────
INPUT_PATH: Path = Path("../data/input")  
MEMORY_PATH: Path = Path("../data/memory")   

# ─────── Config Loading ───────
config = load_config("../configs/config.yaml")

# ─────── Model Configuration ───────
model_path = get_model_path("Meta-Llama-3.1-8B-Instruct-Q8_0.gguf")
if not os.environ.get("MODEL_ARTIFACTS_PATH"):
    model_path = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"

CONTEXT_WINDOW = 8192
MAX_TOKENS = CONTEXT_WINDOW // 8
CHUNK_SIZE = CONTEXT_WINDOW // 2
CHUNK_OVERLAP = CHUNK_SIZE // 8  

EXPERIMENT_NAME = "AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Experiment"
RUN_NAME = "AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Run"
MODEL_NAME = "AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model"


In [10]:
logger.info('Notebook execution started.')

## Verify Assets

In [11]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

In [12]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=model_path,
    asset_name="LLM",
)

# KV Memory

In [13]:
memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
memory.set('dummy key', 'dummy value')

# Load Documents

In [14]:
logger.info("📂 Scanning directory for documents: %s", INPUT_PATH)

supported_extensions = {
".txt": TextLoader,
".csv": lambda path: CSVLoader(path, encoding="utf-8", csv_args={"delimiter": ","}),
".xlsx": UnstructuredExcelLoader,
".docx": UnstructuredWordDocumentLoader,
".pdf": PyPDFLoader,
".md": UnstructuredMarkdownLoader,
}

all_docs = []

for file_path in Path(INPUT_PATH).rglob("*"):
    # Skip hidden/system folders
    if any(part.startswith(".") and part not in {".", ".."} for part in file_path.parts):
        continue
    
    ext = file_path.suffix.lower()
    loader_class = supported_extensions.get(ext)
    
    if loader_class:
        try:
            loader = loader_class(str(file_path))
            docs = loader.load()
            all_docs.extend(docs)
            logger.info("✅ Loaded %d docs from %s", len(docs), file_path.name)
        except Exception as e:
            logger.warning("❌ Failed to load %s: %s", file_path.name, e)
    else:
        logger.info("⚠️ Unsupported file type: %s", file_path.name)

short text: "docx test". Defaulting to English.


short text: "md test". Defaulting to English.


short text: "excel test excel test". Defaulting to English.


In [15]:
INPUT_TEXT = '\n\n'.join([doc.page_content for doc in all_docs])

# MLflow Registration

In [16]:
# 1. Set MLflow tracking URI and experiment
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

2026/04/16 00:29:37 INFO mlflow.tracking.fluent: Experiment with name 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Experiment' does not exist. Creating a new experiment.


Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Experiment


In [17]:
# Define model input/output schema for MLflow signature
input_schema = Schema([
    ColSpec("string", "topic"),
    ColSpec("string", "question"), 
    ColSpec("string", "input_text")
])
output_schema = Schema([
    ColSpec("string", "answer"),
    ColSpec("string", "messages")
])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("✅ Model signature created")

✅ Model signature created


In [18]:
%%time

# === Start MLflow run, log, and register ===
with mlflow.start_run(run_name=RUN_NAME) as run:
    print(f"🚀 Started MLflow run: {run.info.run_id}")

    # Log and register the model using the new universal Logger
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path="../configs/config.yaml",
        docs_path="../data/input",
        model_path=model_path,
        demo_folder="../demo"
    )

    # Construct the URI for the logged model
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"

    # Register the model into MLflow Model Registry
    mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )

logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered.")


🚀 Started MLflow run: dcdba83c428a458eb48f87f239015f87


Successfully registered model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.
2026/04/16 00:33:44 WARNING mlflow.tracking._model_registry.fluent: Run with id dcdba83c428a458eb48f87f239015f87 has no artifacts at artifact path 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model', registering model based on models:/m-21bbd70ab6b141c2861250b4b3659c4f instead
Created version '1' of model 'AIStudio-Agentic-Customer-Feedback-Analyzer-with-LangGraph-Model'.


CPU times: user 1.2 s, sys: 24.5 s, total: 25.7 s
Wall time: 4min 7s


In [19]:
# 3. Retrieve the latest version from the Model Registry
client = MlflowClient()
versions = client.get_latest_versions(MODEL_NAME, stages=["None"])

if not versions:
    raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
    
latest_version = versions[0].version
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

logger.info(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
logger.info(f"Signature: {model_info.signature}")

In [20]:
%%time

# 4. Load the model from the Model Registry
loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
logger.info(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

llama_context: n_ctx_per_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
short text: "docx test". Defaulting to English.
short text: "md test". Defaulting to English.
short text: "excel test excel test". Defaulting to English.


CPU times: user 1.03 s, sys: 2.49 s, total: 3.52 s
Wall time: 1min 6s


In [21]:
# 5. Run a sample inference using the loaded model
input_payload = [{"topic": TOPIC, "question": QUESTION, "input_text": INPUT_TEXT, }]

print("\n=== Running Sample Inference ===")
results = loaded_model.predict(input_payload)
result = results.iloc[0]


=== Running Sample Inference ===


🔁 Processing each chunk: 100%|██████████| 2/2 [00:05<00:00,  2.72s/it, group=✅ Chunk 2 response length: 219 chars]


🔁 Processing each grouped chunk answers: 100%|██████████| 1/1 [00:04<00:00,  4.45s/it, group=🧠 Synthesized partial answer (1/1)]



🔚 === Final Answer ===

# 🧠 Synthesized partial answer (1/1)

**Individuals Providing Feedback on Focus Flow**

The individuals mentioned in the document as providing feedback on the Focus Flow are:

1. James T. (Product Manager, SaaS Company)
2. Aria L. (Freelance UX Designer)
3. Daniel M. (Student, Computer Science)
4. Priya S. (Operations Lead, E-commerce)
5. Nathan C. (Indie App Developer)
6. Linh V. (Marketing Executive)
7. Omar B. (Sales Director)
8. Alina K. (AI Researcher)
9. Marcos E. (Non-profit Volunteer Coordinator)
10. Zoe R. (Remote Freelancer, Content Writer)




# Generated Answer

In [22]:
display(Markdown(result.answer))

# 🧠 Synthesized partial answer (1/1)

**Individuals Providing Feedback on Focus Flow**
===========

The individuals mentioned in the document as providing feedback on the Focus Flow are:

1. James T. (Product Manager, SaaS Company)
2. Aria L. (Freelance UX Designer)
3. Daniel M. (Student, Computer Science)
4. Priya S. (Operations Lead, E-commerce)
5. Nathan C. (Indie App Developer)
6. Linh V. (Marketing Executive)
7. Omar B. (Sales Director)
8. Alina K. (AI Researcher)
9. Marcos E. (Non-profit Volunteer Coordinator)
10. Zoe R. (Remote Freelancer, Content Writer)

# Message History

In [23]:
print(result.messages)

[
    {
        "role": "developer",
        "content": "User submitted a question."
    },
    {
        "role": "user",
        "content": "Which poeple provided the feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Relevance check result:"
    },
    {
        "role": "assistant",
        "content": "yes"
    },
    {
        "role": "developer",
        "content": "\ud83e\udded No cached answer found for question: 'Which poeple provided the feedback?'"
    },
    {
        "role": "developer",
        "content": "\u270f\ufe0f Rewritten user question:"
    },
    {
        "role": "assistant",
        "content": "Who are the individuals mentioned in the document as providing feedback?"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde9 Chunked 1 documents into 2 chunks (size=4096, overlap=256)"
    },
    {
        "role": "developer",
        "content": "\ud83e\udde0 Processed 2 chunks for question: 'Who are the individual

In [24]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).